# Occupational accidents demo

In [0]:
pip install statfin

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
# Import libraries
import statfin
import pandas as pd

# API root object
db = statfin.StatFin()

# Level named ttap (Työtapaturmat) inside StatFin
print(db.StatFin.ttap)  

# Occupational Accidents, 2008-2024
tbl = db.StatFin.ttap["129s.px"]
print(tbl)
# print(tbl.Timeperiod_y)

statfin.PxWebAPI
  url: https://statfin.stat.fi/PXWeb/api/v1/fi/StatFin/ttap
  title: Työtapaturmat
  index:
    (t) 129s.px 129s -- Työtapaturmat, 2008-2024                                                                           
    (t) 129t.px 129t -- Maatalousyrittäjien työtapaturmat 100 000 maatalousyrittäjää kohden, 2000-2024                     
    (t) 12ay.px 12ay -- Palkansaajien ja yrittäjien (pl. maatalousyrittäjät) työtapaturmat sattumispaikan mukaan, 2008-2024
    (t) 12b6.px 12b6 -- Palkansaajien työpaikkatapaturmaan vaikuttanut työsuoritus tapahtumahetkellä, 2008-2024            
    (t) 12b9.px 12b9 -- Palkansaajien työpaikkatapaturmaan johtanut poikkeama, 2008-2024                                   
    (t) 12zz.px 12zz -- Palkansaajien työpaikkatapaturmat 100 000 palkansaajaa kohden, 2018-2024                           
    (t) 13h8.px 13h8 -- Palkansaajien työpaikkatapaturmissa vahingoittuneet ruumiinosat, 2008-2024                         
     ... and 3 more
sta

In [0]:
import requests
import pandas as pd

# This section retrieves accident statistics from the StatFin PXWeb API. 
# The requests are batched by year to avoid exceeding the API request limit.

url = "https://statfin.stat.fi/PXWeb/api/v1/en/StatFin/ttap/129s.px"

years = [str(year) for year in range(2008, 2025)]

all_dfs = []

for year in years:

    print(f"Fetching {year}...")

    query = {
        "query": [
            {
                "code": "Timeperiod_y",
                "selection": {
                    "filter": "item",
                    "values": [year]
                }
            },
            {
                "code": "ammattiasema_13_19960101",
                "selection": {
                    "filter": "item",
                    "values": [
                        "0",
                        "1",
                        "3"
                    ]
                }
            },
            {
                "code": "henkilovahinko_47_19960101",
                "selection": {
                    "filter": "item",
                    "values": [
                        "4-",
                        "0-3",
                        "4-6",
                        "7-14",
                        "15-30",
                        "31-90",
                        "91-180",
                        "181-",
                        "997",
                        "998"
                    ]
                }
            },
            {
                "code": "toimiala_79_20180101",
                "selection": {
                    "filter": "item",
                    "values": [
                        "A", "A01", "A02", "A03",
                        "B", "B05", "B06", "B07", "B08", "B09",
                        "C", "C10", "C11", "C12", "C13", "C14",
                        "C15", "C16", "C17", "C18", "C19", "C20",
                        "C21", "C22", "C23", "C24", "C25", "C26",
                        "C27", "C28", "C29", "C30", "C31", "C32",
                        "C33",
                        "D", "D35",
                        "E", "E36", "E37", "E38", "E39",
                        "F", "F41", "F42", "F43",
                        "G", "G45", "G46", "G47",
                        "H", "H49", "H50", "H51", "H52", "H53",
                        "I", "I55", "I56",
                        "J", "J58", "J59", "J60", "J61", "J62", "J63",
                        "K", "K64", "K65", "K66",
                        "L", "L68",
                        "M", "M69", "M70", "M71", "M72", "M73",
                        "M74", "M75",
                        "N", "N77", "N78", "N79", "N80", "N81", "N82",
                        "O", "O84",
                        "P", "P85",
                        "Q", "Q86", "Q87", "Q88",
                        "R", "R90", "R91", "R92", "R93",
                        "S", "S94", "S95", "S96",
                        "T", "T97", "T98",
                        "U", "U99",
                        "S1313",
                        "X"
                    ]
                }
            },
            {
                "code": "ikaryhma_10_20180101",
                "selection": {
                    "filter": "item",
                    "values": [
                        "15-24",
                        "25-34",
                        "35-44",
                        "45-54",
                        "55-64",
                        "X"
                    ]
                }
            },
            {
                "code": "sukupuoli_9_20180101",
                "selection": {
                    "filter": "item",
                    "values": [
                        "1",
                        "2"
                    ]
                }
            },
            {
                "code": "contentscode",
                "selection": {
                    "filter": "item",
                    "values": [
                        "Tapaturmia"
                    ]
                }
            }
        ],
        "response": {
            "format": "json-stat2"
        }
    }

    # POST request
    resp = requests.post(
        url,
        json=query,
        timeout=120
    )

    # 200 = POST worked
    print(f"{year}: HTTP {resp.status_code}")

    # Show StatFin response if request failed
    if not resp.ok:
        print(resp.text)
        resp.raise_for_status()

    data = resp.json()

    # Convert JSON-stat2 cube to DataFrame
    names = data["id"]

    categories = [
        list(
            data["dimension"][code]["category"]["label"].values()
        )
        for code in names
    ]

    index = pd.MultiIndex.from_product(
        categories,
        names=names
    )

    df_year = pd.DataFrame(
        {
            "value": pd.array(
                data["value"],
                dtype="Float64"
            )
        },
        index=index
    ).reset_index()

    print(f"{year}: {len(df_year):,} rows")

    # Save this year's DataFrame
    all_dfs.append(df_year)


# Combine all years
df = pd.concat(
    all_dfs,
    ignore_index=True
)

print()
print(f"Total rows: {len(df):,}")
print(df.shape)

display(df.head())

Fetching 2008...
2008: HTTP 200
2008: 39,960 rows
Fetching 2009...
2009: HTTP 200
2009: 39,960 rows
Fetching 2010...
2010: HTTP 200
2010: 39,960 rows
Fetching 2011...
2011: HTTP 200
2011: 39,960 rows
Fetching 2012...
2012: HTTP 200
2012: 39,960 rows
Fetching 2013...
2013: HTTP 200
2013: 39,960 rows
Fetching 2014...
2014: HTTP 200
2014: 39,960 rows
Fetching 2015...
2015: HTTP 200
2015: 39,960 rows
Fetching 2016...
2016: HTTP 200
2016: 39,960 rows
Fetching 2017...
2017: HTTP 200
2017: 39,960 rows
Fetching 2018...
2018: HTTP 200
2018: 39,960 rows
Fetching 2019...
2019: HTTP 200
2019: 39,960 rows
Fetching 2020...
2020: HTTP 200
2020: 39,960 rows
Fetching 2021...
2021: HTTP 200
2021: 39,960 rows
Fetching 2022...
2022: HTTP 200
2022: 39,960 rows
Fetching 2023...
2023: HTTP 200
2023: 39,960 rows
Fetching 2024...
2024: HTTP 200
2024: 39,960 rows

Total rows: 679,320
(679320, 8)


Timeperiod_y,ammattiasema_13_19960101,henkilovahinko_47_19960101,toimiala_79_20180101,ikaryhma_10_20180101,sukupuoli_9_20180101,contentscode,value
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",15 - 24,Males,Accidents at work,63.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",15 - 24,Females,Accidents at work,27.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",25 - 34,Males,Accidents at work,101.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",25 - 34,Females,Accidents at work,37.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",35 - 44,Males,Accidents at work,116.0


In [0]:
# Checking data for duplicates, null values, data types etc

df.head()
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 679320 entries, 0 to 679319
Data columns (total 8 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Timeperiod_y                679320 non-null  object 
 1   ammattiasema_13_19960101    679320 non-null  object 
 2   henkilovahinko_47_19960101  679320 non-null  object 
 3   toimiala_79_20180101        679320 non-null  object 
 4   ikaryhma_10_20180101        679320 non-null  object 
 5   sukupuoli_9_20180101        679320 non-null  object 
 6   contentscode                679320 non-null  object 
 7   value                       164143 non-null  Float64
dtypes: Float64(1), object(7)
memory usage: 42.1+ MB


Timeperiod_y                       0
ammattiasema_13_19960101           0
henkilovahinko_47_19960101         0
toimiala_79_20180101               0
ikaryhma_10_20180101               0
sukupuoli_9_20180101               0
contentscode                       0
value                         515177
dtype: int64

# Bronze table

In [0]:
# Raw statfin data, original data untouched
# Converting to spark 
bronze_df = spark.createDataFrame(df)

display(bronze_df)

bronze_df.printSchema()
# timeperiod_y = string -> we want to treat this as a year
# value = double --> the number of accidents is a whole-number count, so we want to change this to an integer

Timeperiod_y,ammattiasema_13_19960101,henkilovahinko_47_19960101,toimiala_79_20180101,ikaryhma_10_20180101,sukupuoli_9_20180101,contentscode,value
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",15 - 24,Males,Accidents at work,63.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",15 - 24,Females,Accidents at work,27.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",25 - 34,Males,Accidents at work,101.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",25 - 34,Females,Accidents at work,37.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",35 - 44,Males,Accidents at work,116.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",35 - 44,Females,Accidents at work,44.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",45 - 54,Males,Accidents at work,200.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",45 - 54,Females,Accidents at work,58.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",55 - 64,Males,Accidents at work,156.0
2008,Wage and salary earners,At least 4 days,"A Agriculture, forestry and fishing (01-03)",55 - 64,Females,Accidents at work,32.0


root
 |-- Timeperiod_y: string (nullable = true)
 |-- ammattiasema_13_19960101: string (nullable = true)
 |-- henkilovahinko_47_19960101: string (nullable = true)
 |-- toimiala_79_20180101: string (nullable = true)
 |-- ikaryhma_10_20180101: string (nullable = true)
 |-- sukupuoli_9_20180101: string (nullable = true)
 |-- contentscode: string (nullable = true)
 |-- value: double (nullable = true)



In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_statfin_occupational_accidents")

In [0]:
%sql
SELECT *
FROM bronze_statfin_occupational_accidents
LIMIT 100;

Timeperiod_y,ammattiasema_13_19960101,henkilovahinko_47_19960101,toimiala_79_20180101,ikaryhma_10_20180101,sukupuoli_9_20180101,contentscode,value
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",25 - 34,Females,Accidents at work,null
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",35 - 44,Males,Accidents at work,3.0
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",35 - 44,Females,Accidents at work,1.0
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",45 - 54,Males,Accidents at work,null
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",45 - 54,Females,Accidents at work,2.0
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",55 - 64,Males,Accidents at work,null
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",55 - 64,Females,Accidents at work,null
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",Unknown,Males,Accidents at work,null
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",Unknown,Females,Accidents at work,null
2010,Wage and salary earners,7-14 days,80 Security and investigation activities,15 - 24,Males,Accidents at work,17.0


# Silver table

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_statfin_occupational_accidents")

silver_df = (
    bronze_df
    .withColumnRenamed("Timeperiod_y", "year")
    .withColumnRenamed(
        "ammattiasema_13_19960101",
        "occupational_status"
    )
    .withColumnRenamed(
        "henkilovahinko_47_19960101",
        "injury_category"
    )
    .withColumnRenamed(
        "toimiala_79_20180101",
        "industry"
    )
    .withColumnRenamed(
        "ikaryhma_10_20180101",
        "age_group"
    )
    .withColumnRenamed(
        "sukupuoli_9_20180101",
        "gender"
    )
    .withColumnRenamed(
        "contentscode",
        "measure"
    )
    .withColumnRenamed(
        "value",
        "accidents"
    )
)

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "year",
        F.col("year").cast("int")
    )
    .withColumn(
        "accidents",
        F.col("accidents").cast("long")
    )
)

display(silver_df)
silver_df.printSchema()

year,occupational_status,injury_category,industry,age_group,gender,measure,accidents
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",25 - 34,Females,Accidents at work,null
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",35 - 44,Males,Accidents at work,3
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",35 - 44,Females,Accidents at work,1
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",45 - 54,Males,Accidents at work,null
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",45 - 54,Females,Accidents at work,2
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",55 - 64,Males,Accidents at work,null
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",55 - 64,Females,Accidents at work,null
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",Unknown,Males,Accidents at work,null
2010,Wage and salary earners,7-14 days,"79 Travel agency, tour operator and other reservation service and related activities",Unknown,Females,Accidents at work,null
2010,Wage and salary earners,7-14 days,80 Security and investigation activities,15 - 24,Males,Accidents at work,17


root
 |-- year: integer (nullable = true)
 |-- occupational_status: string (nullable = true)
 |-- injury_category: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- accidents: long (nullable = true)



In [0]:
# Nulls
silver_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
]).show()

# Duplicate rows
duplicates = (
    silver_df
    .groupBy(
        "year",
        "occupational_status",
        "injury_category",
        "industry",
        "age_group",
        "gender",
        "measure"
    )
    .count()
    .filter(F.col("count") > 1)
)

display(duplicates)

# Looking at distinct variable values
display(
    silver_df
    .select("gender")
    .distinct()
)

# In injury_category there are possibly overlapping values "At least 4 days" and then "4-6 days", "7-14 days" etc 
display(
    silver_df
    .select("injury_category")
    .distinct()
)

# In industry there are main categories and sub categories that need to taken into consideration when creating aggregated tables
display(
    silver_df
    .select("industry")
    .distinct()
)

+----+-------------------+---------------+--------+---------+------+-------+---------+
|year|occupational_status|injury_category|industry|age_group|gender|measure|accidents|
+----+-------------------+---------------+--------+---------+------+-------+---------+
|   0|                  0|              0|       0|        0|     0|      0|   515177|
+----+-------------------+---------------+--------+---------+------+-------+---------+



year,occupational_status,injury_category,industry,age_group,gender,measure,count


gender
Males
Females


injury_category
4-6 days
0-3 days
91-180 days
Fatal accidents
181 days or more
Workers' compensation pension
15-30 days
7-14 days
At least 4 days
31-90 days


industry
23 Manufacture of other non-metallic mineral products
42 Civil engineering
50 Water transport
51 Air transport
I Accommodation and food service activities (55-56)
"M Professional, scientific and technical activities (69-75)"
B Mining and quarrying (05-09)
"29 Manufacture of motor vehicles, trailers and semi-trailers"
"35 Electricity, gas, steam and air conditioning supply"
"36 Water collection, treatment and supply"


In [0]:
# Remove the overlapping value of 'At least 4 days'

silver_df = silver_df.filter(
    F.lower(F.trim(F.col("injury_category")))
    != "at least 4 days"
)

# Verify the value is gone
silver_df.select("injury_category").distinct().show(
    truncate=False
)

+-----------------------------+
|injury_category              |
+-----------------------------+
|4-6 days                     |
|0-3 days                     |
|91-180 days                  |
|Fatal accidents              |
|181 days or more             |
|Workers' compensation pension|
|15-30 days                   |
|7-14 days                    |
|31-90 days                   |
+-----------------------------+



In [0]:
silver_df = silver_df.withColumn(
    "injury_category_sort",
    F.when(F.col("injury_category") == "0-3 days", 1)
     .when(F.col("injury_category") == "4-6 days", 2)
     .when(F.col("injury_category") == "7-14 days", 3)
     .when(F.col("injury_category") == "15-30 days", 4)
     .when(F.col("injury_category") == "31-90 days", 5)
     .when(F.col("injury_category") == "91-180 days", 6)
     .when(F.col("injury_category") == "181 days or more", 7)
     .when(F.col("injury_category") == "Workers' compensation pension", 8)
     .when(F.col("injury_category") == "Fatal accidents", 9)
)

In [0]:
from pyspark.sql import functions as F


# ---------------------------------------------------------
# 1. Keep:
#    - normal two-digit detailed industries
#    - S1313 Local government -sector
#    - Unknown
# ---------------------------------------------------------

silver_df = silver_df.filter(
    F.col("industry").rlike(r"^[0-9]{2}\s")
    | F.col("industry").startswith("S1313")
    | (F.col("industry") == "Unknown")
)


# ---------------------------------------------------------
# 3. Create sub-industry code
#
# Examples:
# "42 Civil engineering"            -> "42"
# "S1313 Local government -sector" -> "S1313"
# "Unknown"                        -> "Unknown"
# ---------------------------------------------------------

silver_df = silver_df.withColumn(
    "industry_code_sub",
    F.when(
        F.col("industry") == "Unknown",
        "Unknown"
    )
    .when(
        F.col("industry").startswith("S1313"),
        "S1313"
    )
    .otherwise(
        F.regexp_extract(
            F.col("industry"),
            r"^([0-9]{2})\s",
            1
        )
    )
)


# ---------------------------------------------------------
# 4. Create sub-industry name
# ---------------------------------------------------------

silver_df = silver_df.withColumn(
    "industry_name_sub",
    F.when(
        F.col("industry") == "Unknown",
        "Unknown"
    )
    .when(
        F.col("industry").startswith("S1313"),
        "Local government -sector"
    )
    .otherwise(
        F.regexp_replace(
            F.col("industry"),
            r"^[0-9]{2}\s+",
            ""
        )
    )
)


# ---------------------------------------------------------
# 5. Create main-industry code
#
# S1313 and Unknown get their own main-industry codes.
# Normal industries are mapped from their two-digit code.
# ---------------------------------------------------------

industry_number = F.col("industry_code_sub").cast("int")

silver_df = silver_df.withColumn(
    "industry_code_main",

    F.when(
        F.col("industry_code_sub") == "Unknown",
        "Unknown"
    )
    .when(
        F.col("industry_code_sub") == "S1313",
        "S1313"
    )
    .when(industry_number.between(1, 3), "A")
    .when(industry_number.between(5, 9), "B")
    .when(industry_number.between(10, 33), "C")
    .when(industry_number == 35, "D")
    .when(industry_number.between(36, 39), "E")
    .when(industry_number.between(41, 43), "F")
    .when(industry_number.between(45, 47), "G")
    .when(industry_number.between(49, 53), "H")
    .when(industry_number.between(55, 56), "I")
    .when(industry_number.between(58, 63), "J")
    .when(industry_number.between(64, 66), "K")
    .when(industry_number == 68, "L")
    .when(industry_number.between(69, 75), "M")
    .when(industry_number.between(77, 82), "N")
    .when(industry_number == 84, "O")
    .when(industry_number == 85, "P")
    .when(industry_number.between(86, 88), "Q")
    .when(industry_number.between(90, 93), "R")
    .when(industry_number.between(94, 96), "S")
    .when(industry_number.between(97, 98), "T")
    .when(industry_number == 99, "U")
)


# ---------------------------------------------------------
# 6. Create main-industry name
# ---------------------------------------------------------

silver_df = silver_df.withColumn(
    "industry_name_main",

    F.when(
        F.col("industry_code_main") == "Unknown",
        "Unknown"
    )
    .when(
        F.col("industry_code_main") == "S1313",
        "Local government -sector"
    )
    .when(
        F.col("industry_code_main") == "A",
        "Agriculture, forestry and fishing"
    )
    .when(
        F.col("industry_code_main") == "B",
        "Mining and quarrying"
    )
    .when(
        F.col("industry_code_main") == "C",
        "Manufacturing"
    )
    .when(
        F.col("industry_code_main") == "D",
        "Electricity, gas, steam and air conditioning supply"
    )
    .when(
        F.col("industry_code_main") == "E",
        "Water supply; sewerage, waste management and remediation activities"
    )
    .when(
        F.col("industry_code_main") == "F",
        "Construction"
    )
    .when(
        F.col("industry_code_main") == "G",
        "Wholesale and retail trade; repair of motor vehicles and motorcycles"
    )
    .when(
        F.col("industry_code_main") == "H",
        "Transportation and storage"
    )
    .when(
        F.col("industry_code_main") == "I",
        "Accommodation and food service activities"
    )
    .when(
        F.col("industry_code_main") == "J",
        "Information and communication"
    )
    .when(
        F.col("industry_code_main") == "K",
        "Financial and insurance activities"
    )
    .when(
        F.col("industry_code_main") == "L",
        "Real estate activities"
    )
    .when(
        F.col("industry_code_main") == "M",
        "Professional, scientific and technical activities"
    )
    .when(
        F.col("industry_code_main") == "N",
        "Administrative and support service activities"
    )
    .when(
        F.col("industry_code_main") == "O",
        "Public administration and defence; compulsory social security"
    )
    .when(
        F.col("industry_code_main") == "P",
        "Education"
    )
    .when(
        F.col("industry_code_main") == "Q",
        "Human health and social work activities"
    )
    .when(
        F.col("industry_code_main") == "R",
        "Arts, entertainment and recreation"
    )
    .when(
        F.col("industry_code_main") == "S",
        "Other service activities"
    )
    .when(
        F.col("industry_code_main") == "T",
        "Activities of households as employers; undifferentiated goods- and services-producing activities of households for own use"
    )
    .when(
        F.col("industry_code_main") == "U",
        "Activities of extraterritorial organisations and bodies"
    )
)


In [0]:
silver_df = silver_df.select(
    "year",
    "occupational_status",
    "injury_category",
    "injury_category_sort",
    "industry_code_main",
    "industry_code_sub",
    "industry_name_main",
    "industry_name_sub",
    "age_group",
    "gender",
    "accidents"
)

In [0]:
display(
    silver_df
    .select(
        "industry_code_main",
        "industry_code_sub",
        "industry_name_main",
        "industry_name_sub"
    )
    .distinct()
    .orderBy("industry_code_sub")
)

industry_code_main,industry_code_sub,industry_name_main,industry_name_sub
A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities"
A,02,"Agriculture, forestry and fishing",Forestry and logging
A,03,"Agriculture, forestry and fishing",Fishing and aquaculture
B,05,Mining and quarrying,Mining of coal and lignite
B,06,Mining and quarrying,Extraction of crude petroleum and natural gas
B,07,Mining and quarrying,Mining of metal ores
B,08,Mining and quarrying,Other mining and quarrying
B,09,Mining and quarrying,Mining support service activities
C,10,Manufacturing,Manufacture of food products
C,11,Manufacturing,Manufacture of beverages


In [0]:
# If it doesn't return zero rows then one of the sub industry codes isn't covered by the mapping
display(
    silver_df
    .filter(F.col("industry_code_main").isNull())
)

year,occupational_status,injury_category,injury_category_sort,industry_code_main,industry_code_sub,industry_name_main,industry_name_sub,age_group,gender,accidents


In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_occupational_accidents_v2")
)

In [0]:
%sql

SELECT *
FROM silver_occupational_accidents_v2
LIMIT 100;

year,occupational_status,injury_category,injury_category_sort,industry_code_main,industry_code_sub,industry_name_main,industry_name_sub,age_group,gender,accidents
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",15 - 24,Males,44
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",15 - 24,Females,42
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",25 - 34,Males,48
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",25 - 34,Females,39
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",35 - 44,Males,35
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",35 - 44,Females,32
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",45 - 54,Males,39
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",45 - 54,Females,36
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",55 - 64,Males,34
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",55 - 64,Females,31


In [0]:
print("DataFrame rows:", silver_df.count())

DataFrame rows: 495720


In [0]:
%sql

SELECT COUNT(*) AS row_count
FROM silver_occupational_accidents_v2;


row_count
495720


In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_occupational_accidents")
)


In [0]:
%sql

DESCRIBE TABLE silver_occupational_accidents;

col_name,data_type,comment
year,int,null
occupational_status,string,null
injury_category,string,null
injury_category_sort,int,null
industry_code_main,string,null
industry_code_sub,string,null
industry_name_main,string,null
industry_name_sub,string,null
age_group,string,null
gender,string,null


# Gold table

In [0]:
%sql

CREATE OR REPLACE TABLE gold_occupational_accidents AS

SELECT
    year,
    occupational_status,
    injury_category,
    injury_category_sort,
    industry_code_main,
    industry_code_sub,
    industry_name_main,
    industry_name_sub,
    age_group,
    gender,
    accidents

FROM silver_occupational_accidents;

SELECT *
FROM gold_occupational_accidents
LIMIT 100;

year,occupational_status,injury_category,injury_category_sort,industry_code_main,industry_code_sub,industry_name_main,industry_name_sub,age_group,gender,accidents
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",15 - 24,Males,44
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",15 - 24,Females,42
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",25 - 34,Males,48
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",25 - 34,Females,39
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",35 - 44,Males,35
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",35 - 44,Females,32
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",45 - 54,Males,39
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",45 - 54,Females,36
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",55 - 64,Males,34
2008,Wage and salary earners,0-3 days,1,A,01,"Agriculture, forestry and fishing","Crop and animal production, hunting and related service activities",55 - 64,Females,31
